In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from pathlib import Path
from scipy import stats

In [ ]:
DATA_DIR = Path("data/european_airlines")
AIRLINE_FILES = [
    "aeg_greece.xlsx",
    "air_france.xlsx",
    "easyjet_uk.xlsx",
    "iceland_air.xlsx",
    "lufthansa.xlsx",
    "nas.xlsx",
    "ryanair.xlsx",
    "sas.xlsx",
    "tui.xlsx",
    "wizz_air_uk.xlsx",
]
MARKET_FILE = "europa_index.xlsx"
RF_FILE = "nowa_rf.xlsx"

EVENT_DATE = pd.Timestamp("2020-03-11")
ESTIMATION_OFFSETS = (-105, -5)          # business-day offsets relative to the event
EVENT_WINDOWS = {
    "±10": (-10, 10),
    "±5": (-5, 5),
    "±3": (-3, 3),
    "±1": (-1, 1),
}
DAY_COUNT = 252                          # adjust if you prefer 252 or 365

In [ ]:
def read_exchange_history(path: Path) -> pd.Series:
    """Read a Reuters-style price history Excel and return closing prices."""
    preview = pd.read_excel(path, header=None, nrows=80)
    header_row = int(preview.index[preview.iloc[:, 0].astype(str)
                                   .str.contains("Exchange Date")][0])
    df = pd.read_excel(path, header=header_row)[["Exchange Date", "Close"]].dropna()
    df["Exchange Date"] = pd.to_datetime(df["Exchange Date"])
    df["Close"] = pd.to_numeric(df["Close"], errors="coerce")
    return (
        df.dropna()
          .set_index("Exchange Date")
          .sort_index()["Close"]
    )

RF_FILE = "nowa_rf.xlsx"

def load_risk_free(path: Path, day_count: int = DAY_COUNT) -> pd.Series:
    rf = pd.read_excel(path)
    rf = rf.rename(columns={rf.columns[0]: "date", rf.columns[1]: "rate"})
    rf = rf.dropna(subset=["date", "rate"])
    rf["date"] = pd.to_datetime(rf["date"])
    rf["rate"] = pd.to_numeric(rf["rate"], errors="coerce")
    rf = rf.dropna().set_index("date").sort_index()
    rf["rf_daily"] = rf["rate"] / 100 / day_count
    return rf["rf_daily"]


def compute_capm_abnormal(
    close: pd.Series,
    market_returns: pd.Series,
    rf_series: pd.Series,
    event_date: pd.Timestamp,
    est_offsets: tuple[int, int],
) -> pd.DataFrame:
    """Estimate CAPM and return DataFrame with abnormal returns."""
    stock_ret = close.pct_change().dropna().rename("stock_return")
    aligned = (
        pd.concat(
            [stock_ret, market_returns.rename("market_return"), rf_series.rename("rf")],
            axis=1,
            join="inner",
        )
        .dropna()
    )
    aligned["excess_stock"] = aligned["stock_return"] - aligned["rf"]
    aligned["excess_market"] = aligned["market_return"] - aligned["rf"]

    est_start = event_date + pd.offsets.BDay(est_offsets[0])
    est_end = event_date + pd.offsets.BDay(est_offsets[1])
    estimation = aligned.loc[est_start:est_end]

    X = sm.add_constant(estimation["excess_market"])
    model = sm.OLS(estimation["excess_stock"], X).fit()
    alpha, beta = model.params["const"], model.params["excess_market"]

    aligned["abnormal"] = aligned["excess_stock"] - (alpha + beta * aligned["excess_market"])
    return aligned

def extract_window(series: pd.Series, event_date: pd.Timestamp, start: int, end: int) -> pd.Series:
    idx = pd.bdate_range(event_date + pd.offsets.BDay(start), event_date + pd.offsets.BDay(end))
    window = series.reindex(idx)
    window.index = range(start, end + 1)
    return window


In [ ]:
# Load common inputs
market_prices = read_exchange_history(DATA_DIR / MARKET_FILE)
market_returns = market_prices.pct_change().dropna()
rf_daily = load_risk_free(DATA_DIR / RF_FILE, day_count=DAY_COUNT)

window_panels: dict[str, pd.DataFrame] = {label: [] for label in EVENT_WINDOWS}
aar_curves: dict[str, pd.Series] = {}
caar_curves: dict[str, pd.Series] = {}
skipped = {}

for fname in AIRLINE_FILES:
    try:
        close = read_exchange_history(DATA_DIR / fname)
        abnormal_df = compute_capm_abnormal(close, market_returns, rf_daily,
                                            EVENT_DATE, ESTIMATION_OFFSETS)
        firm = Path(fname).stem
        for label, (start, end) in EVENT_WINDOWS.items():
            window_panels[label].append(
                extract_window(abnormal_df["abnormal"], EVENT_DATE, start, end).rename(firm)
            )
    except Exception as exc:
        skipped[fname] = str(exc)

summary_rows = []
for label, series_list in window_panels.items():
    window_df = pd.concat(series_list, axis=1)
    aar = window_df.mean(axis=1)
    aar_curves[label] = aar
    caar_curves[label] = aar.cumsum()

    car = window_df.sum(axis=0)
    n = car.count()
    caar_total = car.mean()
    std = car.std(ddof=1)
    t_stat = np.nan if n < 2 or std == 0 else caar_total / (std / np.sqrt(n))
    p_value = np.nan if n < 2 or std == 0 else stats.t.sf(abs(t_stat), df=n - 1) * 2

    summary_rows.append(
        {"window": label, "firms": n, "CAAR": caar_total, "t": t_stat, "p": p_value, "event_AAR": aar.loc[0]}
    )
    window_panels[label] = window_df  # store the concatenated frame

summary = pd.DataFrame(summary_rows).set_index("window").sort_index()

if skipped:
    print("Skipped series:", skipped)


In [ ]:
# Pretty print key outputs
summary_view = summary.copy()
summary_view[["CAAR", "event_AAR"]] *= 100
summary_view = summary_view.round({"CAAR": 2, "event_AAR": 2, "t": 3, "p": 3})
print("CAAR t-tests (values in % where noted):")
print(summary_view)

print("\nEvent-day abnormal returns (%):")
event_day = window_panels["±10"].loc[[0]].T.rename(columns={0: "AR_t0"})
event_day["AR_t0"] = (event_day["AR_t0"] * 100).round(2)
print(event_day.sort_values("AR_t0"))

# To inspect the full AAR/CAAR curves, e.g.:
# print(aar_curves["±10"])
# print(caar_curves["±10"])